<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/07_Adaptive_Utility_Based_Selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# NOTEBOOK 07.0 — ENVIRONMENT AND CONFIGURATION
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import warnings
import gc

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07")
print("ADAPTIVE UTILITY-BASED STRATEGY SELECTION")
print("=" * 100)


# ------------------------------------------------------------
# Google Drive
# ------------------------------------------------------------

from google.colab import drive

DRIVE_ROOT = Path("/content/drive")

if not (DRIVE_ROOT / "MyDrive").exists():
    drive.mount(str(DRIVE_ROOT), force_remount=False)

PROJECT_ROOT = (
    DRIVE_ROOT
    / "MyDrive"
    / "AIR_LLM_Research"
)

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"AIR-LLM project not found:\n{PROJECT_ROOT}"
    )


# ------------------------------------------------------------
# Core configuration
# ------------------------------------------------------------

MASTER_SEED = 42

DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]

TARGET_REGISTRY = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted"
}


# ------------------------------------------------------------
# Input directories
# ------------------------------------------------------------

NOTEBOOK_04_DIR = (
    PROJECT_ROOT
    / "data"
    / "notebook_04"
)

NOTEBOOK_05_DIR = (
    PROJECT_ROOT
    / "data"
    / "notebook_05"
)

NOTEBOOK_06_DIR = (
    PROJECT_ROOT
    / "data"
    / "notebook_06_candidate_evaluation"
)

NOTEBOOK_07_DIR = (
    PROJECT_ROOT
    / "data"
    / "notebook_07"
)


# ------------------------------------------------------------
# Output directories
# ------------------------------------------------------------

PROFILE_DIR = (
    NOTEBOOK_04_DIR
    / "profiles"
)

RECOMMENDATION_DIR = (
    NOTEBOOK_05_DIR
    / "recommendations"
)

EVALUATION_DIR = (
    NOTEBOOK_06_DIR
    / "outputs"
    / "results"
)

OUTPUT_DIR = (
    NOTEBOOK_07_DIR
)

SELECTION_DIR = (
    OUTPUT_DIR
    / "selection"
)

AUDIT_DIR = (
    OUTPUT_DIR
    / "audit"
)

METADATA_DIR = (
    OUTPUT_DIR
    / "metadata"
)

DIAGNOSTIC_DIR = (
    OUTPUT_DIR
    / "diagnostics"
)


for directory in [
    OUTPUT_DIR,
    SELECTION_DIR,
    AUDIT_DIR,
    METADATA_DIR,
    DIAGNOSTIC_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# Adaptive utility configuration
# ------------------------------------------------------------

CRITERIA = [
    "accuracy",
    "distribution",
    "dependency",
    "predictive",
    "efficiency"
]

BASE_WEIGHTS = {
    "accuracy": 0.35,
    "distribution": 0.20,
    "dependency": 0.20,
    "predictive": 0.15,
    "efficiency": 0.10
}


# ------------------------------------------------------------
# Confidence / abstention configuration
# ------------------------------------------------------------

CONFIDENCE_THRESHOLD = 0.55
MARGIN_THRESHOLD = 0.05
STABILITY_THRESHOLD = 0.60

MIN_EVIDENCE_ROWS = 2


print("\nPROJECT")
print("-" * 100)
print(f"Project root : {PROJECT_ROOT}")
print(f"Notebook 04  : {NOTEBOOK_04_DIR}")
print(f"Notebook 05  : {NOTEBOOK_05_DIR}")
print(f"Notebook 06  : {NOTEBOOK_06_DIR}")
print(f"Notebook 07  : {NOTEBOOK_07_DIR}")

print("\nDATASETS")
print("-" * 100)

for dataset_id in DATASETS:
    print(
        f"{dataset_id:20s} | "
        f"Target: {TARGET_REGISTRY[dataset_id]}"
    )

print("\nUTILITY WEIGHTS")
print("-" * 100)

for key, value in BASE_WEIGHTS.items():
    print(f"{key:15s}: {value:.3f}")

print("\nNotebook 07 environment initialized.")

NOTEBOOK 07.1 — LOAD MASKED VALIDATION DATA
Project root : /content/drive/MyDrive/AIR_LLM
Masked data  : /content/drive/MyDrive/AIR_LLM/validation/masked
⚠ adult_income: masked validation file not found
⚠ bank_marketing: masked validation file not found
⚠ diabetes_130us: masked validation file not found


RuntimeError: 
No masked validation datasets were found.

Expected files such as:

adult_income_masked_validation.csv
bank_marketing_masked_validation.csv
diabetes_130us_masked_validation.csv

Place them under:

PROJECT_ROOT/validation/masked/


In [ ]:
# ============================================================
# NOTEBOOK 07.1 — ARTIFACT DISCOVERY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.1")
print("ARTIFACT DISCOVERY")
print("=" * 100)


def find_first_existing(candidates):
    for path in candidates:
        if path.exists() and path.is_file():
            return path
    return None


# ------------------------------------------------------------
# Notebook 04
# ------------------------------------------------------------

FEATURE_PROFILE_PATH = find_first_existing([
    PROFILE_DIR / "feature" / "feature_profile.csv",
    PROFILE_DIR / "feature_profile.csv"
])

MISSINGNESS_PROFILE_PATH = find_first_existing([
    PROFILE_DIR / "feature" / "missingness_profile.csv",
    PROFILE_DIR / "missingness_profile.csv"
])

DATASET_PROFILE_PATH = find_first_existing([
    PROFILE_DIR / "dataset" / "dataset_profile.csv",
    PROFILE_DIR / "dataset_profile.csv"
])


# ------------------------------------------------------------
# Notebook 05
# ------------------------------------------------------------

LLM_RECOMMENDATION_PATH = find_first_existing([
    RECOMMENDATION_DIR / "llm_recommendations.csv",
    RECOMMENDATION_DIR / "recommendations.csv",
    RECOMMENDATION_DIR / "recommendation_summary.csv"
])


# ------------------------------------------------------------
# Notebook 06
# ------------------------------------------------------------

EVALUATION_CANDIDATES = [
    EVALUATION_DIR / "candidate_evidence.csv",
    EVALUATION_DIR / "candidate_evaluation_results.csv",
    EVALUATION_DIR / "final_empirical_winners.csv",
    NOTEBOOK_06_DIR / "candidate_evidence.csv",
    NOTEBOOK_06_DIR / "candidate_evaluation_results.csv"
]

EVALUATION_PATH = find_first_existing(
    EVALUATION_CANDIDATES
)


print("\nARTIFACT STATUS")
print("-" * 100)

artifacts = {
    "Feature profile":
        FEATURE_PROFILE_PATH,

    "Missingness profile":
        MISSINGNESS_PROFILE_PATH,

    "Dataset profile":
        DATASET_PROFILE_PATH,

    "LLM recommendations":
        LLM_RECOMMENDATION_PATH,

    "Candidate evaluation":
        EVALUATION_PATH
}

for name, path in artifacts.items():

    print(
        f"{name:25s} : "
        f"{'FOUND' if path else 'MISSING'}"
    )

    if path:
        print(
            f"{'':25s}   {path}"
        )


if FEATURE_PROFILE_PATH is None:
    raise FileNotFoundError(
        "Notebook 04 feature profile was not found."
    )

if MISSINGNESS_PROFILE_PATH is None:
    raise FileNotFoundError(
        "Notebook 04 missingness profile was not found."
    )

if EVALUATION_PATH is None:
    raise FileNotFoundError(
        "Notebook 06 candidate evaluation evidence "
        "was not found."
    )

In [ ]:
# ============================================================
# NOTEBOOK 07.2 — LOAD INPUT ARTIFACTS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.2")
print("LOADING PROFILES AND EMPIRICAL EVIDENCE")
print("=" * 100)


FEATURE_PROFILE_DF = pd.read_csv(
    FEATURE_PROFILE_PATH
)

MISSINGNESS_PROFILE_DF = pd.read_csv(
    MISSINGNESS_PROFILE_PATH
)

if DATASET_PROFILE_PATH:
    DATASET_PROFILE_DF = pd.read_csv(
        DATASET_PROFILE_PATH
    )
else:
    DATASET_PROFILE_DF = pd.DataFrame()


EVALUATION_DF = pd.read_csv(
    EVALUATION_PATH
)


if LLM_RECOMMENDATION_PATH:

    LLM_RECOMMENDATION_DF = pd.read_csv(
        LLM_RECOMMENDATION_PATH
    )

else:

    LLM_RECOMMENDATION_DF = pd.DataFrame()


print("\nLOADED")
print("-" * 100)

print(
    f"Feature profiles       : "
    f"{len(FEATURE_PROFILE_DF):,}"
)

print(
    f"Missingness profiles   : "
    f"{len(MISSINGNESS_PROFILE_DF):,}"
)

print(
    f"Dataset profiles       : "
    f"{len(DATASET_PROFILE_DF):,}"
)

print(
    f"Evaluation evidence    : "
    f"{len(EVALUATION_DF):,}"
)

print(
    f"LLM recommendations    : "
    f"{len(LLM_RECOMMENDATION_DF):,}"
)

In [ ]:
# ============================================================
# NOTEBOOK 07.3 — INPUT SCHEMA NORMALIZATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.3")
print("INPUT SCHEMA NORMALIZATION")
print("=" * 100)


def normalize_columns(df):

    df = df.copy()

    df.columns = [
        str(c)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
        for c in df.columns
    ]

    return df


FEATURE_PROFILE_DF = normalize_columns(
    FEATURE_PROFILE_DF
)

MISSINGNESS_PROFILE_DF = normalize_columns(
    MISSINGNESS_PROFILE_DF
)

DATASET_PROFILE_DF = normalize_columns(
    DATASET_PROFILE_DF
)

EVALUATION_DF = normalize_columns(
    EVALUATION_DF
)

LLM_RECOMMENDATION_DF = normalize_columns(
    LLM_RECOMMENDATION_DF
)


print("\nEVALUATION COLUMNS")
print("-" * 100)

for column in EVALUATION_DF.columns:
    print(column)

In [ ]:
# ============================================================
# NOTEBOOK 07.4 — EVALUATION SCHEMA ADAPTER
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.4")
print("EVALUATION SCHEMA ADAPTER")
print("=" * 100)


def find_column(df, candidates):

    for candidate in candidates:

        candidate = candidate.lower()

        if candidate in df.columns:
            return candidate

    return None


COLUMN_MAP = {

    "dataset_id": find_column(
        EVALUATION_DF,
        [
            "dataset_id",
            "dataset",
            "data_set"
        ]
    ),

    "feature": find_column(
        EVALUATION_DF,
        [
            "feature",
            "feature_name",
            "target_feature",
            "column"
        ]
    ),

    "strategy": find_column(
        EVALUATION_DF,
        [
            "strategy",
            "strategy_name",
            "method",
            "candidate_strategy",
            "candidate"
        ]
    ),

    "mechanism": find_column(
        EVALUATION_DF,
        [
            "mechanism",
            "missingness_mechanism"
        ]
    ),

    "rate": find_column(
        EVALUATION_DF,
        [
            "rate",
            "missingness_rate",
            "requested_rate"
        ]
    ),

    "repetition": find_column(
        EVALUATION_DF,
        [
            "repetition",
            "repeat",
            "rep"
        ]
    ),

    "accuracy": find_column(
        EVALUATION_DF,
        [
            "accuracy",
            "accuracy_score",
            "imputation_accuracy",
            "mae",
            "rmse",
            "error"
        ]
    ),

    "distribution": find_column(
        EVALUATION_DF,
        [
            "distribution",
            "distribution_score",
            "distribution_fidelity",
            "ks_statistic",
            "wasserstein_distance"
        ]
    ),

    "dependency": find_column(
        EVALUATION_DF,
        [
            "dependency",
            "dependency_score",
            "dependency_preservation",
            "correlation_error"
        ]
    ),

    "predictive": find_column(
        EVALUATION_DF,
        [
            "predictive",
            "predictive_utility",
            "predictive_score",
            "task_utility"
        ]
    ),

    "efficiency": find_column(
        EVALUATION_DF,
        [
            "efficiency",
            "efficiency_score",
            "runtime",
            "runtime_seconds",
            "time_seconds"
        ]
    )
}


print("\nCOLUMN MAPPING")
print("-" * 100)

for key, value in COLUMN_MAP.items():

    print(
        f"{key:15s} -> "
        f"{value if value else 'NOT FOUND'}"
    )


required = [
    "dataset_id",
    "feature",
    "strategy"
]

missing_required = [
    key
    for key in required
    if COLUMN_MAP[key] is None
]

if missing_required:

    raise ValueError(
        "Notebook 06 evaluation evidence is missing "
        "required columns:\n"
        + "\n".join(missing_required)
    )

In [ ]:
# ============================================================
# NOTEBOOK 07.5 — STANDARDIZE EMPIRICAL EVIDENCE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.5")
print("STANDARDIZING EMPIRICAL EVIDENCE")
print("=" * 100)


STANDARD_EVIDENCE = pd.DataFrame()

STANDARD_EVIDENCE["dataset_id"] = (
    EVALUATION_DF[
        COLUMN_MAP["dataset_id"]
    ].astype(str)
)

STANDARD_EVIDENCE["feature"] = (
    EVALUATION_DF[
        COLUMN_MAP["feature"]
    ].astype(str)
)

STANDARD_EVIDENCE["strategy"] = (
    EVALUATION_DF[
        COLUMN_MAP["strategy"]
    ].astype(str)
)


# ------------------------------------------------------------
# Optional experimental dimensions
# ------------------------------------------------------------

for name in [
    "mechanism",
    "rate",
    "repetition"
]:

    source = COLUMN_MAP.get(name)

    if source:

        STANDARD_EVIDENCE[name] = (
            EVALUATION_DF[source]
        )

    else:

        STANDARD_EVIDENCE[name] = np.nan


# ------------------------------------------------------------
# Metric extraction
# ------------------------------------------------------------

for criterion in CRITERIA:

    source = COLUMN_MAP.get(
        criterion
    )

    if source:

        STANDARD_EVIDENCE[
            criterion
        ] = pd.to_numeric(
            EVALUATION_DF[source],
            errors="coerce"
        )

    else:

        STANDARD_EVIDENCE[
            criterion
        ] = np.nan


print("\nSTANDARD EVIDENCE")
print("-" * 100)

display(
    STANDARD_EVIDENCE.head(10)
)

print(
    f"\nRows: {len(STANDARD_EVIDENCE):,}"
)

In [ ]:
# ============================================================
# NOTEBOOK 07.6 — METRIC DIRECTION HANDLING
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.6")
print("METRIC DIRECTION NORMALIZATION")
print("=" * 100)


HIGHER_IS_BETTER = {
    "accuracy": True,
    "distribution": True,
    "dependency": True,
    "predictive": True,
    "efficiency": True
}


# ------------------------------------------------------------
# Detect error-style metrics
# ------------------------------------------------------------

def metric_is_error_like(column_name):

    if column_name is None:
        return False

    name = column_name.lower()

    error_terms = [
        "mae",
        "rmse",
        "error",
        "distance",
        "wasserstein",
        "ks_",
        "correlation_error",
        "runtime",
        "time_seconds"
    ]

    return any(
        term in name
        for term in error_terms
    )


for criterion in CRITERIA:

    source = COLUMN_MAP.get(
        criterion
    )

    if metric_is_error_like(source):

        HIGHER_IS_BETTER[
            criterion
        ] = False


print("\nMETRIC DIRECTIONS")
print("-" * 100)

for criterion in CRITERIA:

    direction = (
        "HIGHER"
        if HIGHER_IS_BETTER[criterion]
        else "LOWER"
    )

    print(
        f"{criterion:15s}: "
        f"{direction} IS BETTER"
    )

In [ ]:
# ============================================================
# NOTEBOOK 07.7 — ROBUST METRIC NORMALIZATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.7")
print("ROBUST METRIC NORMALIZATION")
print("=" * 100)


def robust_minmax(series, higher_is_better=True):

    x = pd.to_numeric(
        series,
        errors="coerce"
    )

    valid = x.dropna()

    if len(valid) == 0:
        return pd.Series(
            0.5,
            index=series.index
        )

    q_low = valid.quantile(0.05)
    q_high = valid.quantile(0.95)

    if not np.isfinite(q_low):
        q_low = valid.min()

    if not np.isfinite(q_high):
        q_high = valid.max()

    if q_high <= q_low:

        result = pd.Series(
            0.5,
            index=series.index
        )

    else:

        clipped = x.clip(
            lower=q_low,
            upper=q_high
        )

        result = (
            clipped - q_low
        ) / (
            q_high - q_low
        )

    if not higher_is_better:

        result = 1.0 - result

    return result.fillna(0.5).clip(
        0.0,
        1.0
    )


for criterion in CRITERIA:

    STANDARD_EVIDENCE[
        f"{criterion}_score"
    ] = (
        STANDARD_EVIDENCE
        .groupby(
            [
                "dataset_id",
                "feature"
            ],
            dropna=False
        )[criterion]
        .transform(
            lambda s, c=criterion:
                robust_minmax(
                    s,
                    HIGHER_IS_BETTER[c]
                )
        )
    )


print("Metric normalization completed.")

In [ ]:
# ============================================================
# NOTEBOOK 07.8 — FEATURE CONTEXT
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.8")
print("LOADING FEATURE CONTEXT")
print("=" * 100)


def find_profile_column(df, candidates):

    for candidate in candidates:

        if candidate in df.columns:
            return candidate

    return None


missing_dataset_col = find_profile_column(
    MISSINGNESS_PROFILE_DF,
    ["dataset_id", "dataset"]
)

missing_feature_col = find_profile_column(
    MISSINGNESS_PROFILE_DF,
    ["feature", "feature_name", "column"]
)

missing_rate_col = find_profile_column(
    MISSINGNESS_PROFILE_DF,
    [
        "missing_percentage",
        "missing_percent",
        "missing_rate",
        "missing_fraction"
    ]
)


if (
    missing_dataset_col
    and missing_feature_col
    and missing_rate_col
):

    CONTEXT_DF = (
        MISSINGNESS_PROFILE_DF[
            [
                missing_dataset_col,
                missing_feature_col,
                missing_rate_col
            ]
        ]
        .rename(
            columns={
                missing_dataset_col:
                    "dataset_id",

                missing_feature_col:
                    "feature",

                missing_rate_col:
                    "natural_missing_rate"
            }
        )
    )

else:

    CONTEXT_DF = (
        STANDARD_EVIDENCE[
            [
                "dataset_id",
                "feature"
            ]
        ]
        .drop_duplicates()
    )

    CONTEXT_DF[
        "natural_missing_rate"
    ] = 0.0


CONTEXT_DF[
    "natural_missing_rate"
] = pd.to_numeric(
    CONTEXT_DF[
        "natural_missing_rate"
    ],
    errors="coerce"
).fillna(0.0)


CONTEXT_DF[
    "natural_missing_rate"
] = (
    CONTEXT_DF[
        "natural_missing_rate"
    ]
    .clip(0, 1)
)


STANDARD_EVIDENCE = STANDARD_EVIDENCE.merge(
    CONTEXT_DF,
    on=[
        "dataset_id",
        "feature"
    ],
    how="left"
)


print(
    f"Feature context rows: "
    f"{len(CONTEXT_DF):,}"
)

In [ ]:
# ============================================================
# NOTEBOOK 07.9 — ADAPTIVE UTILITY WEIGHTS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.9")
print("ADAPTIVE UTILITY WEIGHTING")
print("=" * 100)


def adaptive_weights(
    missing_rate,
    dependency_strength=0.5,
    task_relevance=0.5
):

    weights = BASE_WEIGHTS.copy()

    r = float(
        np.clip(
            missing_rate,
            0.0,
            1.0
        )
    )

    dep = float(
        np.clip(
            dependency_strength,
            0.0,
            1.0
        )
    )

    task = float(
        np.clip(
            task_relevance,
            0.0,
            1.0
        )
    )


    # --------------------------------------------------------
    # Higher missingness -> accuracy becomes more important
    # --------------------------------------------------------

    weights["accuracy"] += (
        0.15 * r
    )


    # --------------------------------------------------------
    # Strong dependencies -> dependency preservation matters
    # --------------------------------------------------------

    weights["dependency"] += (
        0.10 * dep
    )


    # --------------------------------------------------------
    # Strong downstream relevance -> predictive utility matters
    # --------------------------------------------------------

    weights["predictive"] += (
        0.10 * task
    )


    # --------------------------------------------------------
    # Efficiency remains relevant, especially for low-burden
    # features
    # --------------------------------------------------------

    weights["efficiency"] -= (
        0.05 * r
    )


    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    total = sum(
        max(v, 0.0)
        for v in weights.values()
    )

    if total <= 0:
        return BASE_WEIGHTS.copy()

    return {
        key:
            max(value, 0.0) / total
        for key, value in weights.items()
    }


# ------------------------------------------------------------
# Estimate dependency/task relevance from available profiles
# ------------------------------------------------------------

def get_context_value(
    dataset_id,
    feature,
    candidates,
    default=0.5
):

    for df in candidates:

        if df is None or df.empty:
            continue

        dataset_col = find_profile_column(
            df,
            ["dataset_id", "dataset"]
        )

        feature_col = find_profile_column(
            df,
            ["feature", "feature_name", "column"]
        )

        if not dataset_col or not feature_col:
            continue

        subset = df[
            (
                df[dataset_col].astype(str)
                == str(dataset_id)
            )
            &
            (
                df[feature_col].astype(str)
                == str(feature)
            )
        ]

        if subset.empty:
            continue

        for column in candidates:

            if column in subset.columns:

                value = pd.to_numeric(
                    subset.iloc[0][column],
                    errors="coerce"
                )

                if pd.notna(value):

                    return float(
                        np.clip(
                            value,
                            0,
                            1
                        )
                    )

    return default


# ------------------------------------------------------------
# Create adaptive weights per feature
# ------------------------------------------------------------

WEIGHT_ROWS = []

for dataset_id, feature in (
    STANDARD_EVIDENCE[
        [
            "dataset_id",
            "feature"
        ]
    ]
    .drop_duplicates()
    .itertuples(index=False)
):

    context = CONTEXT_DF[
        (
            CONTEXT_DF["dataset_id"].astype(str)
            == str(dataset_id)
        )
        &
        (
            CONTEXT_DF["feature"].astype(str)
            == str(feature)
        )
    ]

    if context.empty:
        missing_rate = 0.0
    else:
        missing_rate = float(
            context.iloc[0][
                "natural_missing_rate"
            ]
        )


    dependency_strength = get_context_value(
        dataset_id,
        feature,
        [
            FEATURE_PROFILE_DF
        ],
        default=0.5
    )


    task_relevance = get_context_value(
        dataset_id,
        feature,
        [
            FEATURE_PROFILE_DF
        ],
        default=0.5
    )


    weights = adaptive_weights(
        missing_rate,
        dependency_strength,
        task_relevance
    )


    WEIGHT_ROWS.append({

        "dataset_id":
            dataset_id,

        "feature":
            feature,

        "missing_rate":
            missing_rate,

        "dependency_strength":
            dependency_strength,

        "task_relevance":
            task_relevance,

        **{
            f"weight_{k}":
                v
            for k, v in weights.items()
        }
    })


WEIGHTS_DF = pd.DataFrame(
    WEIGHT_ROWS
)


print(
    f"Adaptive weight profiles: "
    f"{len(WEIGHTS_DF):,}"
)

display(
    WEIGHTS_DF.head(10)
)

In [ ]:
# ============================================================
# NOTEBOOK 07.10 — MULTI-OBJECTIVE UTILITY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.10")
print("MULTI-OBJECTIVE UTILITY CALCULATION")
print("=" * 100)


UTILITY_DF = STANDARD_EVIDENCE.merge(
    WEIGHTS_DF,
    on=[
        "dataset_id",
        "feature"
    ],
    how="left"
)


UTILITY_DF["utility_accuracy"] = (
    UTILITY_DF["accuracy_score"]
    * UTILITY_DF["weight_accuracy"]
)

UTILITY_DF["utility_distribution"] = (
    UTILITY_DF["distribution_score"]
    * UTILITY_DF["weight_distribution"]
)

UTILITY_DF["utility_dependency"] = (
    UTILITY_DF["dependency_score"]
    * UTILITY_DF["weight_dependency"]
)

UTILITY_DF["utility_predictive"] = (
    UTILITY_DF["predictive_score"]
    * UTILITY_DF["weight_predictive"]
)

UTILITY_DF["utility_efficiency"] = (
    UTILITY_DF["efficiency_score"]
    * UTILITY_DF["weight_efficiency"]
)


UTILITY_DF["adaptive_utility"] = (
    UTILITY_DF["utility_accuracy"]
    + UTILITY_DF["utility_distribution"]
    + UTILITY_DF["utility_dependency"]
    + UTILITY_DF["utility_predictive"]
    + UTILITY_DF["utility_efficiency"]
)


UTILITY_DF["adaptive_utility"] = (
    UTILITY_DF["adaptive_utility"]
    .clip(0, 1)
)


print("\nUTILITY SUMMARY")
print("-" * 100)

print(
    UTILITY_DF[
        "adaptive_utility"
    ].describe()
)

display(
    UTILITY_DF[
        [
            "dataset_id",
            "feature",
            "strategy",
            "adaptive_utility"
        ]
    ].head(15)
)

In [ ]:
# ============================================================
# NOTEBOOK 07.11 — ROBUST SCENARIO AGGREGATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.11")
print("ROBUST AGGREGATION ACROSS EXPERIMENTAL CONDITIONS")
print("=" * 100)


GROUP_COLUMNS = [
    "dataset_id",
    "feature",
    "strategy"
]


AGGREGATION_DF = (
    UTILITY_DF
    .groupby(
        GROUP_COLUMNS,
        dropna=False
    )
    .agg(
        mean_utility=(
            "adaptive_utility",
            "mean"
        ),

        median_utility=(
            "adaptive_utility",
            "median"
        ),

        utility_std=(
            "adaptive_utility",
            "std"
        ),

        min_utility=(
            "adaptive_utility",
            "min"
        ),

        max_utility=(
            "adaptive_utility",
            "max"
        ),

        evidence_count=(
            "adaptive_utility",
            "count"
        )
    )
    .reset_index()
)


AGGREGATION_DF[
    "utility_std"
] = AGGREGATION_DF[
    "utility_std"
].fillna(0.0)


# ------------------------------------------------------------
# Robust utility
# ------------------------------------------------------------

AGGREGATION_DF[
    "robust_utility"
] = (
    0.50
    * AGGREGATION_DF["mean_utility"]
    +
    0.30
    * AGGREGATION_DF["median_utility"]
    +
    0.20
    * AGGREGATION_DF["min_utility"]
)


AGGREGATION_DF[
    "robust_utility"
] = (
    AGGREGATION_DF[
        "robust_utility"
    ]
    .clip(0, 1)
)


print(
    f"Candidate-feature combinations: "
    f"{len(AGGREGATION_DF):,}"
)

display(
    AGGREGATION_DF.head(15)
)

In [ ]:
# ============================================================
# NOTEBOOK 07.12 — CROSS-CONDITION STABILITY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.12")
print("CROSS-CONDITION STABILITY")
print("=" * 100)


def calculate_stability(group):

    values = group[
        "adaptive_utility"
    ].dropna()

    if len(values) <= 1:
        return 1.0

    mean_value = values.mean()

    if mean_value <= 1e-12:
        return 0.0

    cv = (
        values.std(ddof=0)
        / mean_value
    )

    stability = 1.0 / (
        1.0 + cv
    )

    return float(
        np.clip(
            stability,
            0,
            1
        )
    )


STABILITY_ROWS = []

for keys, group in UTILITY_DF.groupby(
    GROUP_COLUMNS,
    dropna=False
):

    dataset_id, feature, strategy = keys

    STABILITY_ROWS.append({

        "dataset_id":
            dataset_id,

        "feature":
            feature,

        "strategy":
            strategy,

        "stability":
            calculate_stability(group),

        "condition_count":
            int(
                group[
                    "adaptive_utility"
                ].count()
            )
    })


STABILITY_DF = pd.DataFrame(
    STABILITY_ROWS
)


AGGREGATION_DF = AGGREGATION_DF.merge(
    STABILITY_DF,
    on=GROUP_COLUMNS,
    how="left"
)


print(
    f"Stability profiles: "
    f"{len(STABILITY_DF):,}"
)

In [ ]:
# ============================================================
# NOTEBOOK 07.13 — CANDIDATE RANKING
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.13")
print("EMPIRICAL CANDIDATE RANKING")
print("=" * 100)


AGGREGATION_DF[
    "utility_rank"
] = (
    AGGREGATION_DF
    .groupby(
        [
            "dataset_id",
            "feature"
        ]
    )[
        "robust_utility"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)


AGGREGATION_DF[
    "utility_percentile"
] = (
    AGGREGATION_DF
    .groupby(
        [
            "dataset_id",
            "feature"
        ]
    )[
        "robust_utility"
    ]
    .rank(
        pct=True
    )
)


print(
    "Candidate ranking completed."
)

display(
    AGGREGATION_DF.sort_values(
        [
            "dataset_id",
            "feature",
            "utility_rank"
        ]
    ).head(20)
)

In [ ]:
# ============================================================
# NOTEBOOK 07.14 — SELECTION MARGIN
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.14")
print("BEST-VS-SECOND-BEST MARGIN")
print("=" * 100)


MARGIN_ROWS = []

for keys, group in AGGREGATION_DF.groupby(
    [
        "dataset_id",
        "feature"
    ],
    dropna=False
):

    dataset_id, feature = keys

    ranked = group.sort_values(
        "robust_utility",
        ascending=False
    )

    values = ranked[
        "robust_utility"
    ].to_numpy()

    best = (
        float(values[0])
        if len(values) >= 1
        else 0.0
    )

    second = (
        float(values[1])
        if len(values) >= 2
        else 0.0
    )

    margin = best - second

    MARGIN_ROWS.append({

        "dataset_id":
            dataset_id,

        "feature":
            feature,

        "best_utility":
            best,

        "second_best_utility":
            second,

        "selection_margin":
            margin,

        "candidate_count":
            len(values)
    })


MARGIN_DF = pd.DataFrame(
    MARGIN_ROWS
)


AGGREGATION_DF = AGGREGATION_DF.merge(
    MARGIN_DF,
    on=[
        "dataset_id",
        "feature"
    ],
    how="left"
)


print(
    f"Feature decisions: "
    f"{len(MARGIN_DF):,}"
)

In [ ]:
# ============================================================
# NOTEBOOK 07.15 — EMPIRICAL CONFIDENCE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.15")
print("EMPIRICAL CONFIDENCE ESTIMATION")
print("=" * 100)


def confidence_from_margin(margin):

    return float(
        np.clip(
            margin / 0.25,
            0,
            1
        )
    )


def combine_confidence(
    margin_confidence,
    stability,
    evidence_count
):

    evidence_factor = min(
        evidence_count / 5.0,
        1.0
    )

    confidence = (
        0.45
        * margin_confidence
        +
        0.40
        * stability
        +
        0.15
        * evidence_factor
    )

    return float(
        np.clip(
            confidence,
            0,
            1
        )
    )


BEST_ROWS = []

for keys, group in AGGREGATION_DF.groupby(
    [
        "dataset_id",
        "feature"
    ],
    dropna=False
):

    dataset_id, feature = keys

    ranked = group.sort_values(
        "robust_utility",
        ascending=False
    )

    best = ranked.iloc[0]

    margin_conf = confidence_from_margin(
        best["selection_margin"]
    )

    empirical_confidence = combine_confidence(
        margin_conf,
        best["stability"],
        best["evidence_count"]
    )

    BEST_ROWS.append({

        "dataset_id":
            dataset_id,

        "feature":
            feature,

        "selected_strategy":
            best["strategy"],

        "selected_utility":
            float(
                best["robust_utility"]
            ),

        "selection_margin":
            float(
                best["selection_margin"]
            ),

        "stability":
            float(
                best["stability"]
            ),

        "evidence_count":
            int(
                best["evidence_count"]
            ),

        "empirical_confidence":
            empirical_confidence
    })


EMPIRICAL_SELECTION_DF = pd.DataFrame(
    BEST_ROWS
)


print(
    f"Empirical selections: "
    f"{len(EMPIRICAL_SELECTION_DF):,}"
)

display(
    EMPIRICAL_SELECTION_DF.head(20)
)

In [ ]:
# ============================================================
# NOTEBOOK 07.15 — EMPIRICAL CONFIDENCE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.15")
print("EMPIRICAL CONFIDENCE ESTIMATION")
print("=" * 100)


def confidence_from_margin(margin):

    return float(
        np.clip(
            margin / 0.25,
            0,
            1
        )
    )


def combine_confidence(
    margin_confidence,
    stability,
    evidence_count
):

    evidence_factor = min(
        evidence_count / 5.0,
        1.0
    )

    confidence = (
        0.45
        * margin_confidence
        +
        0.40
        * stability
        +
        0.15
        * evidence_factor
    )

    return float(
        np.clip(
            confidence,
            0,
            1
        )
    )


BEST_ROWS = []

for keys, group in AGGREGATION_DF.groupby(
    [
        "dataset_id",
        "feature"
    ],
    dropna=False
):

    dataset_id, feature = keys

    ranked = group.sort_values(
        "robust_utility",
        ascending=False
    )

    best = ranked.iloc[0]

    margin_conf = confidence_from_margin(
        best["selection_margin"]
    )

    empirical_confidence = combine_confidence(
        margin_conf,
        best["stability"],
        best["evidence_count"]
    )

    BEST_ROWS.append({

        "dataset_id":
            dataset_id,

        "feature":
            feature,

        "selected_strategy":
            best["strategy"],

        "selected_utility":
            float(
                best["robust_utility"]
            ),

        "selection_margin":
            float(
                best["selection_margin"]
            ),

        "stability":
            float(
                best["stability"]
            ),

        "evidence_count":
            int(
                best["evidence_count"]
            ),

        "empirical_confidence":
            empirical_confidence
    })


EMPIRICAL_SELECTION_DF = pd.DataFrame(
    BEST_ROWS
)


print(
    f"Empirical selections: "
    f"{len(EMPIRICAL_SELECTION_DF):,}"
)

display(
    EMPIRICAL_SELECTION_DF.head(20)
)

In [ ]:
# ============================================================
# NOTEBOOK 07.16 — LLM RECOMMENDATION ALIGNMENT
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.16")
print("LLM RECOMMENDATION ALIGNMENT")
print("=" * 100)


LLM_ALIGNMENT_DF = EMPIRICAL_SELECTION_DF.copy()

LLM_ALIGNMENT_DF[
    "llm_recommended_strategy"
] = np.nan

LLM_ALIGNMENT_DF[
    "llm_alignment"
] = 0.0


if not LLM_RECOMMENDATION_DF.empty:

    llm_dataset_col = find_profile_column(
        LLM_RECOMMENDATION_DF,
        [
            "dataset_id",
            "dataset"
        ]
    )

    llm_feature_col = find_profile_column(
        LLM_RECOMMENDATION_DF,
        [
            "feature",
            "feature_name",
            "column"
        ]
    )

    llm_strategy_col = find_profile_column(
        LLM_RECOMMENDATION_DF,
        [
            "strategy",
            "strategy_name",
            "recommended_strategy",
            "candidate_strategy"
        ]
    )

    if (
        llm_dataset_col
        and llm_feature_col
        and llm_strategy_col
    ):

        llm_small = (
            LLM_RECOMMENDATION_DF[
                [
                    llm_dataset_col,
                    llm_feature_col,
                    llm_strategy_col
                ]
            ]
            .rename(
                columns={
                    llm_dataset_col:
                        "dataset_id",

                    llm_feature_col:
                        "feature",

                    llm_strategy_col:
                        "llm_recommended_strategy"
                }
            )
        )


        # First recommendation per feature
        llm_small = (
            llm_small
            .drop_duplicates(
                [
                    "dataset_id",
                    "feature"
                ]
            )
        )


        LLM_ALIGNMENT_DF = (
            LLM_ALIGNMENT_DF.drop(
                columns=[
                    "llm_recommended_strategy"
                ],
                errors="ignore"
            )
            .merge(
                llm_small,
                on=[
                    "dataset_id",
                    "feature"
                ],
                how="left"
            )
        )


        LLM_ALIGNMENT_DF[
            "llm_alignment"
        ] = (
            LLM_ALIGNMENT_DF[
                "selected_strategy"
            ].astype(str).str.lower()
            ==
            LLM_ALIGNMENT_DF[
                "llm_recommended_strategy"
            ].astype(str).str.lower()
        ).astype(float)


print(
    "LLM alignment calculated."
)

print(
    "Empirical evidence remains the primary "
    "selection authority."
)

In [ ]:
# ============================================================
# NOTEBOOK 07.17 — CONFIDENCE AND ABSTENTION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.17")
print("CONFIDENCE AND ABSTENTION")
print("=" * 100)


FINAL_SELECTION_DF = (
    LLM_ALIGNMENT_DF.copy()
)


FINAL_SELECTION_DF[
    "confidence_pass"
] = (
    FINAL_SELECTION_DF[
        "empirical_confidence"
    ]
    >= CONFIDENCE_THRESHOLD
)


FINAL_SELECTION_DF[
    "margin_pass"
] = (
    FINAL_SELECTION_DF[
        "selection_margin"
    ]
    >= MARGIN_THRESHOLD
)


FINAL_SELECTION_DF[
    "stability_pass"
] = (
    FINAL_SELECTION_DF[
        "stability"
    ]
    >= STABILITY_THRESHOLD
)


FINAL_SELECTION_DF[
    "abstain"
] = ~(
    FINAL_SELECTION_DF[
        "confidence_pass"
    ]
    &
    FINAL_SELECTION_DF[
        "margin_pass"
    ]
    &
    FINAL_SELECTION_DF[
        "stability_pass"
    ]
)


FINAL_SELECTION_DF[
    "decision_status"
] = np.where(
    FINAL_SELECTION_DF["abstain"],
    "ABSTAIN_ADDITIONAL_EVALUATION",
    "SELECT"
)


print("\nDECISION SUMMARY")
print("-" * 100)

print(
    FINAL_SELECTION_DF[
        "decision_status"
    ].value_counts()
)

In [ ]:
# ============================================================
# NOTEBOOK 07.18 — ABSTENTION CANDIDATE EXPANSION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.18")
print("ABSTENTION CANDIDATE EXPANSION")
print("=" * 100)


ABSTENTION_ROWS = []


for _, selected in FINAL_SELECTION_DF.iterrows():

    if selected["decision_status"] != "ABSTAIN_ADDITIONAL_EVALUATION":
        continue

    subset = AGGREGATION_DF[
        (
            AGGREGATION_DF["dataset_id"]
            == selected["dataset_id"]
        )
        &
        (
            AGGREGATION_DF["feature"]
            == selected["feature"]
        )
    ].copy()


    subset = subset.sort_values(
        "robust_utility",
        ascending=False
    )


    for _, candidate in subset.head(3).iterrows():

        ABSTENTION_ROWS.append({

            "dataset_id":
                selected["dataset_id"],

            "feature":
                selected["feature"],

            "candidate_strategy":
                candidate["strategy"],

            "robust_utility":
                candidate["robust_utility"],

            "stability":
                candidate["stability"],

            "reason":
                "Low empirical confidence; "
                "candidate should receive additional "
                "evaluation before final acceptance."
        })


ABSTENTION_DF = pd.DataFrame(
    ABSTENTION_ROWS
)


print(
    f"Abstention candidate rows: "
    f"{len(ABSTENTION_DF):,}"
)

if not ABSTENTION_DF.empty:

    display(
        ABSTENTION_DF.head(20)
    )

In [ ]:
# ============================================================
# NOTEBOOK 07.19 — FINAL DECISION MAP
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.19")
print("FINAL STRATEGY DECISION MAP")
print("=" * 100)


DECISION_MAP = {}


for _, row in FINAL_SELECTION_DF.iterrows():

    key = (
        f"{row['dataset_id']}::"
        f"{row['feature']}"
    )

    DECISION_MAP[key] = {

        "dataset_id":
            row["dataset_id"],

        "feature":
            row["feature"],

        "selected_strategy":
            row["selected_strategy"],

        "utility":
            float(
                row["selected_utility"]
            ),

        "empirical_confidence":
            float(
                row["empirical_confidence"]
            ),

        "selection_margin":
            float(
                row["selection_margin"]
            ),

        "stability":
            float(
                row["stability"]
            ),

        "llm_recommended_strategy":
            (
                None
                if pd.isna(
                    row[
                        "llm_recommended_strategy"
                    ]
                )
                else str(
                    row[
                        "llm_recommended_strategy"
                    ]
                )
            ),

        "llm_alignment":
            float(
                row["llm_alignment"]
            ),

        "status":
            row["decision_status"]
    }


DECISION_MAP_PATH = (
    METADATA_DIR
    / "decision_map.json"
)


with open(
    DECISION_MAP_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        DECISION_MAP,
        f,
        indent=2,
        allow_nan=False
    )


print(
    f"Decision map saved:\n"
    f"{DECISION_MAP_PATH}"
)

In [ ]:
# ============================================================
# NOTEBOOK 07.20 — SELECTION AUDIT
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.20")
print("SELECTION AUDIT")
print("=" * 100)


AUDIT_COLUMNS = [

    "dataset_id",
    "feature",
    "selected_strategy",

    "selected_utility",

    "selection_margin",

    "stability",

    "evidence_count",

    "empirical_confidence",

    "llm_recommended_strategy",

    "llm_alignment",

    "confidence_pass",

    "margin_pass",

    "stability_pass",

    "abstain",

    "decision_status"
]


SELECTION_AUDIT_DF = (
    FINAL_SELECTION_DF[
        [
            c
            for c in AUDIT_COLUMNS
            if c in FINAL_SELECTION_DF.columns
        ]
    ]
    .sort_values(
        [
            "dataset_id",
            "feature"
        ]
    )
    .reset_index(drop=True)
)


AUDIT_PATH = (
    AUDIT_DIR
    / "selection_audit.csv"
)


SELECTION_AUDIT_DF.to_csv(
    AUDIT_PATH,
    index=False
)


print(
    f"Audit rows: "
    f"{len(SELECTION_AUDIT_DF):,}"
)

print(
    f"Saved to:\n{AUDIT_PATH}"
)

display(
    SELECTION_AUDIT_DF.head(20)
)

In [ ]:
# ============================================================
# NOTEBOOK 07.21 — ADAPTIVE SELECTION OUTPUT
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.21")
print("ADAPTIVE SELECTION RESULTS")
print("=" * 100)


ADAPTIVE_SELECTION_COLUMNS = [

    "dataset_id",
    "feature",
    "selected_strategy",
    "selected_utility",
    "selection_margin",
    "stability",
    "empirical_confidence",
    "llm_recommended_strategy",
    "llm_alignment",
    "decision_status"
]


ADAPTIVE_SELECTION_DF = (
    FINAL_SELECTION_DF[
        [
            c
            for c in ADAPTIVE_SELECTION_COLUMNS
            if c in FINAL_SELECTION_DF.columns
        ]
    ]
    .sort_values(
        [
            "dataset_id",
            "feature"
        ]
    )
    .reset_index(drop=True)
)


ADAPTIVE_SELECTION_PATH = (
    SELECTION_DIR
    / "adaptive_selection.csv"
)


ADAPTIVE_SELECTION_DF.to_csv(
    ADAPTIVE_SELECTION_PATH,
    index=False
)


print(
    f"Adaptive selections: "
    f"{len(ADAPTIVE_SELECTION_DF):,}"
)

display(
    ADAPTIVE_SELECTION_DF.head(20)
)

In [ ]:
# ============================================================
# NOTEBOOK 07.22 — FINAL STRATEGY SELECTION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.22")
print("FINAL STRATEGY SELECTION")
print("=" * 100)


FINAL_STRATEGY_DF = (
    FINAL_SELECTION_DF[
        [
            "dataset_id",
            "feature",
            "selected_strategy",
            "selected_utility",
            "empirical_confidence",
            "selection_margin",
            "stability",
            "decision_status"
        ]
    ]
    .copy()
)


FINAL_STRATEGY_DF[
    "selection_basis"
] = np.where(
    FINAL_STRATEGY_DF[
        "decision_status"
    ]
    == "SELECT",
    "EMPIRICAL_UTILITY",
    "ABSTAIN_PENDING_ADDITIONAL_EVALUATION"
)


FINAL_STRATEGY_PATH = (
    SELECTION_DIR
    / "final_strategy_selection.csv"
)


FINAL_STRATEGY_DF.to_csv(
    FINAL_STRATEGY_PATH,
    index=False
)


print(
    f"Final strategy records: "
    f"{len(FINAL_STRATEGY_DF):,}"
)

display(
    FINAL_STRATEGY_DF.head(20)
)

In [ ]:
# ============================================================
# NOTEBOOK 07.23 — UTILITY COMPONENT ANALYSIS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.23")
print("UTILITY COMPONENT ANALYSIS")
print("=" * 100)


WINNER_KEYS = FINAL_STRATEGY_DF[
    FINAL_STRATEGY_DF[
        "decision_status"
    ] == "SELECT"
][
    [
        "dataset_id",
        "feature",
        "selected_strategy"
    ]
]


WINNER_COMPONENTS_DF = UTILITY_DF.merge(
    WINNER_KEYS,
    left_on=[
        "dataset_id",
        "feature",
        "strategy"
    ],
    right_on=[
        "dataset_id",
        "feature",
        "selected_strategy"
    ],
    how="inner"
)


if not WINNER_COMPONENTS_DF.empty:

    COMPONENT_COLUMNS = [

        "dataset_id",
        "feature",
        "strategy",

        "accuracy_score",
        "distribution_score",
        "dependency_score",
        "predictive_score",
        "efficiency_score",

        "weight_accuracy",
        "weight_distribution",
        "weight_dependency",
        "weight_predictive",
        "weight_efficiency",

        "adaptive_utility"
    ]


    COMPONENT_COLUMNS = [
        c
        for c in COMPONENT_COLUMNS
        if c in WINNER_COMPONENTS_DF.columns
    ]


    WINNER_COMPONENTS_DF[
        COMPONENT_COLUMNS
    ].to_csv(
        AUDIT_DIR
        / "winner_utility_components.csv",
        index=False
    )


print(
    "Utility component analysis completed."
)

In [ ]:
# ============================================================
# NOTEBOOK 07.24 — STRATEGY SELECTION SUMMARY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.24")
print("STRATEGY SELECTION SUMMARY")
print("=" * 100)


STRATEGY_SUMMARY_DF = (
    FINAL_STRATEGY_DF
    .groupby(
        "selected_strategy",
        dropna=False
    )
    .agg(
        feature_count=(
            "feature",
            "count"
        ),

        mean_utility=(
            "selected_utility",
            "mean"
        ),

        mean_confidence=(
            "empirical_confidence",
            "mean"
        ),

        mean_margin=(
            "selection_margin",
            "mean"
        ),

        mean_stability=(
            "stability",
            "mean"
        )
    )
    .reset_index()
    .sort_values(
        "feature_count",
        ascending=False
    )
)


STRATEGY_SUMMARY_PATH = (
    DIAGNOSTIC_DIR
    / "strategy_selection_summary.csv"
)


STRATEGY_SUMMARY_DF.to_csv(
    STRATEGY_SUMMARY_PATH,
    index=False
)


display(
    STRATEGY_SUMMARY_DF
)

In [ ]:
# ============================================================
# NOTEBOOK 07.25 — DATASET SELECTION SUMMARY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.25")
print("DATASET-LEVEL SELECTION SUMMARY")
print("=" * 100)


DATASET_SELECTION_SUMMARY_DF = (
    FINAL_STRATEGY_DF
    .groupby(
        "dataset_id"
    )
    .agg(
        features=(
            "feature",
            "count"
        ),

        selected=(
            "decision_status",
            lambda s:
                int(
                    (s == "SELECT").sum()
                )
        ),

        abstained=(
            "decision_status",
            lambda s:
                int(
                    (
                        s
                        == "ABSTAIN_ADDITIONAL_EVALUATION"
                    ).sum()
                )
        ),

        mean_utility=(
            "selected_utility",
            "mean"
        ),

        mean_confidence=(
            "empirical_confidence",
            "mean"
        ),

        mean_stability=(
            "stability",
            "mean"
        )
    )
    .reset_index()
)


DATASET_SUMMARY_PATH = (
    DIAGNOSTIC_DIR
    / "dataset_selection_summary.csv"
)


DATASET_SELECTION_SUMMARY_DF.to_csv(
    DATASET_SUMMARY_PATH,
    index=False
)


display(
    DATASET_SELECTION_SUMMARY_DF
)

In [ ]:
# ============================================================
# NOTEBOOK 07.26 — FINAL VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.26")
print("FINAL SELECTION VALIDATION")
print("=" * 100)


VALIDATION_ROWS = []


# ------------------------------------------------------------
# Dataset coverage
# ------------------------------------------------------------

for dataset_id in DATASETS:

    count = int(
        (
            FINAL_STRATEGY_DF[
                "dataset_id"
            ]
            == dataset_id
        ).sum()
    )

    VALIDATION_ROWS.append({

        "check":
            f"Dataset coverage: {dataset_id}",

        "passed":
            count > 0,

        "value":
            count
    })


# ------------------------------------------------------------
# Utility bounds
# ------------------------------------------------------------

utility_valid = (
    FINAL_STRATEGY_DF[
        "selected_utility"
    ]
    .between(0, 1)
    .all()
)

VALIDATION_ROWS.append({

    "check":
        "Utility values in [0,1]",

    "passed":
        bool(utility_valid),

    "value":
        float(
            FINAL_STRATEGY_DF[
                "selected_utility"
            ].min()
        )
})


# ------------------------------------------------------------
# Confidence bounds
# ------------------------------------------------------------

confidence_valid = (
    FINAL_STRATEGY_DF[
        "empirical_confidence"
    ]
    .between(0, 1)
    .all()
)

VALIDATION_ROWS.append({

    "check":
        "Confidence values in [0,1]",

    "passed":
        bool(confidence_valid),

    "value":
        float(
            FINAL_STRATEGY_DF[
                "empirical_confidence"
            ].min()
        )
})


# ------------------------------------------------------------
# Stability bounds
# ------------------------------------------------------------

stability_valid = (
    FINAL_STRATEGY_DF[
        "stability"
    ]
    .between(0, 1)
    .all()
)

VALIDATION_ROWS.append({

    "check":
        "Stability values in [0,1]",

    "passed":
        bool(stability_valid),

    "value":
        float(
            FINAL_STRATEGY_DF[
                "stability"
            ].min()
        )
})


# ------------------------------------------------------------
# Strategy non-null
# ------------------------------------------------------------

strategy_valid = (
    FINAL_STRATEGY_DF[
        "selected_strategy"
    ]
    .notna()
    .all()
)

VALIDATION_ROWS.append({

    "check":
        "Selected strategy available",

    "passed":
        bool(strategy_valid),

    "value":
        int(
            FINAL_STRATEGY_DF[
                "selected_strategy"
            ].notna().sum()
        )
})


# ------------------------------------------------------------
# No target selection
# ------------------------------------------------------------

target_failures = []

for _, row in FINAL_STRATEGY_DF.iterrows():

    target = TARGET_REGISTRY[
        row["dataset_id"]
    ]

    if str(
        row["feature"]
    ) == str(target):

        target_failures.append(
            row["dataset_id"]
        )


VALIDATION_ROWS.append({

    "check":
        "Target variables excluded",

    "passed":
        len(target_failures) == 0,

    "value":
        len(target_failures)
})


VALIDATION_DF = pd.DataFrame(
    VALIDATION_ROWS
)


VALIDATION_PATH = (
    DIAGNOSTIC_DIR
    / "notebook_07_validation.csv"
)


VALIDATION_DF.to_csv(
    VALIDATION_PATH,
    index=False
)


display(
    VALIDATION_DF
)


if not VALIDATION_DF[
    "passed"
].all():

    raise RuntimeError(
        "Notebook 07 validation failed."
    )


print(
    "\nALL NOTEBOOK 07 SELECTION VALIDATIONS PASSED."
)

In [ ]:
# ============================================================
# NOTEBOOK 07.27 — DRIVE PERSISTENCE VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.27")
print("DRIVE PERSISTENCE VALIDATION")
print("=" * 100)


REQUIRED_OUTPUTS = [

    ADAPTIVE_SELECTION_PATH,

    FINAL_STRATEGY_PATH,

    AUDIT_PATH,

    DECISION_MAP_PATH,

    STRATEGY_SUMMARY_PATH,

    DATASET_SUMMARY_PATH,

    VALIDATION_PATH
]


PERSISTENCE_ROWS = []


for path in REQUIRED_OUTPUTS:

    exists = (
        path.exists()
        and path.is_file()
        and path.stat().st_size > 0
    )

    PERSISTENCE_ROWS.append({

        "file":
            str(path),

        "exists":
            exists,

        "size_bytes":
            path.stat().st_size
            if path.exists()
            else 0
    })


PERSISTENCE_DF = pd.DataFrame(
    PERSISTENCE_ROWS
)


display(
    PERSISTENCE_DF
)


if not PERSISTENCE_DF[
    "exists"
].all():

    raise RuntimeError(
        "One or more Notebook 07 output files "
        "failed persistence validation."
    )


print(
    "\nDRIVE PERSISTENCE VALIDATION PASSED."
)

In [ ]:
# ============================================================
# NOTEBOOK 07.28 — DECISION EXPLANATIONS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.28")
print("AUDITABLE DECISION EXPLANATIONS")
print("=" * 100)


EXPLANATION_ROWS = []


for _, row in FINAL_STRATEGY_DF.iterrows():

    dataset_id = row[
        "dataset_id"
    ]

    feature = row[
        "feature"
    ]

    strategy = row[
        "selected_strategy"
    ]

    status = row[
        "decision_status"
    ]


    context = CONTEXT_DF[
        (
            CONTEXT_DF[
                "dataset_id"
            ].astype(str)
            == str(dataset_id)
        )
        &
        (
            CONTEXT_DF[
                "feature"
            ].astype(str)
            == str(feature)
        )
    ]


    if context.empty:

        missing_rate = 0.0

    else:

        missing_rate = float(
            context.iloc[0][
                "natural_missing_rate"
            ]
        )


    if status == "SELECT":

        decision_text = (
            f"{strategy} selected because it achieved "
            f"the highest robust empirical utility "
            f"({row['selected_utility']:.4f}) "
            f"with empirical confidence "
            f"{row['empirical_confidence']:.4f} "
            f"and stability "
            f"{row['stability']:.4f}."
        )

    else:

        decision_text = (
            f"No strategy was accepted automatically "
            f"for {feature}; empirical confidence "
            f"({row['empirical_confidence']:.4f}) "
            f"or cross-condition stability "
            f"({row['stability']:.4f}) was insufficient."
        )


    EXPLANATION_ROWS.append({

        "dataset_id":
            dataset_id,

        "feature":
            feature,

        "missing_rate":
            missing_rate,

        "selected_strategy":
            strategy,

        "empirical_utility":
            row["selected_utility"],

        "empirical_confidence":
            row["empirical_confidence"],

        "selection_margin":
            row["selection_margin"],

        "stability":
            row["stability"],

        "llm_recommended_strategy":
            row[
                "llm_recommended_strategy"
            ],

        "llm_alignment":
            row["llm_alignment"],

        "decision_status":
            status,

        "decision_explanation":
            decision_text
    })


DECISION_EXPLANATION_DF = pd.DataFrame(
    EXPLANATION_ROWS
)


EXPLANATION_PATH = (
    AUDIT_DIR
    / "decision_explanations.csv"
)


DECISION_EXPLANATION_DF.to_csv(
    EXPLANATION_PATH,
    index=False
)


display(
    DECISION_EXPLANATION_DF.head(20)
)

In [ ]:
# ============================================================
# NOTEBOOK 07.29 — MANIFEST
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07.29")
print("NOTEBOOK MANIFEST")
print("=" * 100)


def file_hash(path):

    if not path.exists():
        return None

    sha256 = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda:
                f.read(1024 * 1024),
            b""
        ):

            sha256.update(chunk)

    return sha256.hexdigest()


OUTPUT_MANIFEST = {}


for path in REQUIRED_OUTPUTS + [
    EXPLANATION_PATH
]:

    if path.exists():

        OUTPUT_MANIFEST[
            str(path.relative_to(PROJECT_ROOT))
        ] = {

            "size_bytes":
                path.stat().st_size,

            "sha256":
                file_hash(path)
        }


MANIFEST = {

    "notebook":
        "07_Adaptive_Utility_Based_Selection",

    "framework":
        "AIR-LLM",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "master_seed":
        MASTER_SEED,

    "datasets":
        DATASETS,

    "target_registry":
        TARGET_REGISTRY,

    "utility_criteria":
        CRITERIA,

    "base_weights":
        BASE_WEIGHTS,

    "confidence_threshold":
        CONFIDENCE_THRESHOLD,

    "margin_threshold":
        MARGIN_THRESHOLD,

    "stability_threshold":
        STABILITY_THRESHOLD,

    "input_artifacts": {

        "feature_profile":
            str(FEATURE_PROFILE_PATH),

        "missingness_profile":
            str(MISSINGNESS_PROFILE_PATH),

        "candidate_evaluation":
            str(EVALUATION_PATH),

        "llm_recommendations":
            (
                str(LLM_RECOMMENDATION_PATH)
                if LLM_RECOMMENDATION_PATH
                else None
            )
    },

    "results": {

        "candidate_combinations":
            len(AGGREGATION_DF),

        "feature_decisions":
            len(FINAL_STRATEGY_DF),

        "automatic_selections":
            int(
                (
                    FINAL_STRATEGY_DF[
                        "decision_status"
                    ]
                    == "SELECT"
                ).sum()
            ),

        "abstentions":
            int(
                (
                    FINAL_STRATEGY_DF[
                        "decision_status"
                    ]
                    == "ABSTAIN_ADDITIONAL_EVALUATION"
                ).sum()
            )
    },

    "outputs":
        OUTPUT_MANIFEST
}


MANIFEST_PATH = (
    METADATA_DIR
    / "notebook_07_manifest.json"
)


with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        MANIFEST,
        f,
        indent=2,
        allow_nan=False
    )


print(
    f"Manifest saved:\n{MANIFEST_PATH}"
)

In [ ]:
# ============================================================
# NOTEBOOK 07.30 — FINAL NOTEBOOK STATUS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 07 COMPLETE")
print("=" * 100)


TOTAL_FEATURES = len(
    FINAL_STRATEGY_DF
)

TOTAL_SELECTED = int(
    (
        FINAL_STRATEGY_DF[
            "decision_status"
        ]
        == "SELECT"
    ).sum()
)

TOTAL_ABSTAINED = int(
    (
        FINAL_STRATEGY_DF[
            "decision_status"
        ]
        == "ABSTAIN_ADDITIONAL_EVALUATION"
    ).sum()
)


print("\nEXPERIMENTAL SELECTION")
print("-" * 100)

print(
    f"Datasets              : {len(DATASETS)}"
)

print(
    f"Feature decisions     : {TOTAL_FEATURES}"
)

print(
    f"Automatically selected: {TOTAL_SELECTED}"
)

print(
    f"Abstained             : {TOTAL_ABSTAINED}"
)


print("\nSELECTION QUALITY")
print("-" * 100)

print(
    "Mean empirical utility : "
    f"{FINAL_STRATEGY_DF['selected_utility'].mean():.4f}"
)

print(
    "Mean empirical confidence: "
    f"{FINAL_STRATEGY_DF['empirical_confidence'].mean():.4f}"
)

print(
    "Mean stability         : "
    f"{FINAL_STRATEGY_DF['stability'].mean():.4f}"
)

print(
    "Mean selection margin  : "
    f"{FINAL_STRATEGY_DF['selection_margin'].mean():.4f}"
)


print("\nVALIDATION")
print("-" * 100)

print(
    "Utility validation     : PASSED"
)

print(
    "Confidence validation  : PASSED"
)

print(
    "Stability validation   : PASSED"
)

print(
    "Target exclusion       : PASSED"
)

print(
    "Drive persistence      : PASSED"
)


print("\nOUTPUTS")
print("-" * 100)

print(
    f"Adaptive selection:\n"
    f"{ADAPTIVE_SELECTION_PATH}"
)

print(
    f"Final strategy:\n"
    f"{FINAL_STRATEGY_PATH}"
)

print(
    f"Selection audit:\n"
    f"{AUDIT_PATH}"
)

print(
    f"Decision map:\n"
    f"{DECISION_MAP_PATH}"
)

print(
    f"Decision explanations:\n"
    f"{EXPLANATION_PATH}"
)

print(
    f"Manifest:\n"
    f"{MANIFEST_PATH}"
)


print("\n" + "=" * 100)

if TOTAL_ABSTAINED > 0:

    print(
        "NOTEBOOK 07 COMPLETED WITH CONTROLLED ABSTENTIONS"
    )

    print(
        "Abstained features require additional candidate "
        "evaluation before final acceptance."
    )

else:

    print(
        "ALL FEATURES RECEIVED EMPIRICALLY SUPPORTED "
        "STRATEGY SELECTIONS."
    )

print(
    "NOTEBOOK 07 — ADAPTIVE UTILITY-BASED SELECTION COMPLETE"
)

print(
    "NEXT: NOTEBOOK 08 — FINAL IMPUTATION AND END-TO-END EVALUATION"
)

print("=" * 100)


gc.collect()